In [ ]:
"""
🟡 B-2. 전이(轉移) — 오늘 만든 것을 **새 데이터에 그대로** 옮겨 본다  📩 (응용 · 목표 30분)

오늘 우리는 IMDB 영화 리뷰로 RNN 분류기를 만들었다. 그런데 이게 IMDB 전용일까?
**아니다.** 같은 파이프라인을 문자메시지 스팸 분류에 그대로 붙여 본다.

    원문 → 토큰화 → 사전 → 인코딩 → 앞쪽 패딩 → Embedding → RNN → Linear → sigmoid
    (바뀌는 건 '어떤 데이터를 넣느냐' 뿐이다)

이 미션이 진짜로 확인하는 것 세 가지:
  ① 오늘 만든 도구가 **다른 문제에도 통하는가** (통한다)
  ② 문장이 **짧으면** RNN이 더 잘하는가 (오늘 §7의 뒷면)
  ③ ★ 정확도 숫자에 **속지 않는 법** — 불균형 데이터의 함정

데이터: SMS Spam Collection (UCI). HuggingFace `ucirvine/sms_spam`, 5,574건.
"""

In [ ]:
import sys
from pathlib import Path
try:
    _HERE = Path(__file__).resolve().parent
except NameError:                       # 노트북 셀에는 __file__ 이 없다
    _HERE = Path.cwd()
sys.path.insert(0, str(_HERE.parent))   # day03/ 모듈 재사용

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset

from textutils import tokenize_en, build_vocab, pad_and_tensor   # ← 어제·오늘 그대로
from model import IMDBRnn                                        # ← 이름만 IMDB, 일반 RNN 분류기

torch.manual_seed(42)

VOCAB_SIZE, MAX_LEN = 2000, 30    # ★ MAX_LEN 이 100 → 30. 문자는 짧으니까
EMBED, HIDDEN, BATCH, LR = 32, 32, 64, 1e-3

## 1. 데이터 — 문자메시지 5,574건

`label` 0 = 정상(ham), 1 = 스팸. split이 train 하나뿐이라 직접 나눈다.

In [ ]:
ds = load_dataset("ucirvine/sms_spam")["train"].shuffle(seed=42)
n_train = 4500
train, val = ds.select(range(n_train)), ds.select(range(n_train, len(ds)))

train_toks = [tokenize_en(t) for t in train["sms"]]
val_toks = [tokenize_en(t) for t in val["sms"]]
y_train = torch.tensor([int(x) for x in train["label"]], dtype=torch.float32)
y_val = torch.tensor([int(x) for x in val["label"]], dtype=torch.float32)

print(f"훈련 {len(train)} · 검증 {len(val)}")
print(f"검증셋 스팸 비율: {y_val.mean():.1%}   ← 기억해 둘 것! (③에서 쓴다)")
lens = sorted(len(t) for t in train_toks)
print(f"문자 길이(토큰): 중앙값 {lens[len(lens)//2]}  ← IMDB는 179였다")

## 2. 파이프라인 — 어제·오늘 함수를 **한 줄도 안 고치고** 쓴다

In [ ]:
word2idx, _ = build_vocab(train_toks, max_size=VOCAB_SIZE)
X_train = pad_and_tensor(train_toks, word2idx, MAX_LEN)
X_val = pad_and_tensor(val_toks, word2idx, MAX_LEN)
print("X_train:", tuple(X_train.shape), "· 사전:", len(word2idx))

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH, shuffle=True)

## 3. 훈련 — 모델도 그대로

In [ ]:
model = IMDBRnn(len(word2idx), EMBED, HIDDEN)
opt = torch.optim.Adam(model.parameters(), lr=LR)
crit = nn.BCELoss()

for epoch in range(8):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = ((model(X_val) > 0.5).float() == y_val).float().mean().item()
    print(f"에포크 {epoch+1}: 검증 정확도 {acc:.4f}")

## 4. ★ 정확도에 속지 않기 — 불균형 데이터의 함정

정확도가 높게 나왔을 것이다. 그런데 **검증셋의 스팸은 13% 남짓**이다.
그러면 아무 생각 없이 **전부 '정상'이라고만 찍어도** 정확도가 87% 가까이 나온다.
그래서 정확도 하나만 보면 안 된다. **스팸을 실제로 잡았는지**를 따로 봐야 한다.

- **재현율(recall)**: 진짜 스팸 중 몇 %를 잡아냈나 → 놓친 스팸
- **정밀도(precision)**: 스팸이라 한 것 중 몇 %가 진짜였나 → 정상 메일을 스팸함에 넣은 사고

In [ ]:
model.eval()
with torch.no_grad():
    prob = model(X_val)
pred = (prob > 0.5).float()

acc = (pred == y_val).float().mean().item()
baseline = 1 - y_val.mean().item()          # 전부 '정상'이라 찍었을 때의 정확도
tp = ((pred == 1) & (y_val == 1)).sum().item()
fp = ((pred == 1) & (y_val == 0)).sum().item()
fn = ((pred == 0) & (y_val == 1)).sum().item()
recall = tp / (tp + fn) if tp + fn else 0.0
precision = tp / (tp + fp) if tp + fp else 0.0

print(f"정확도            : {acc:.4f}")
print(f"전부 '정상' 베이스라인: {baseline:.4f}   ← 이것보다 높아야 의미가 있다")
print(f"스팸 재현율(recall)  : {recall:.4f}   (진짜 스팸 {tp+fn}건 중 {tp}건 잡음)")
print(f"스팸 정밀도(precision): {precision:.4f}   (정상을 스팸이라 한 사고 {fp}건)")

## 5. 내 문자로 시험해 보기

스팸 문구를 직접 지어 넣어 보자. 어떤 단어가 스팸 판정을 끌어올리는가?

In [ ]:
def is_spam(text):
    x = pad_and_tensor([tokenize_en(text)], word2idx, MAX_LEN)
    with torch.no_grad():
        p = model(x).item()
    print(f"  [{'스팸' if p > 0.5 else '정상'} {p:.2f}] {text}")
    return p


print("\n내 문자 시험")
is_spam("hey are we still meeting for lunch tomorrow")
is_spam("WINNER!! You have won a FREE prize. Text WIN to 80086 to claim now")
is_spam("call me when you get home")
is_spam("URGENT! Your account will be suspended. Click the link to verify")

## ★ 두 가지 더 관찰할 것

**(가) 훈련 초반 정확도를 다시 보자.** 에포크 1의 정확도가 위 '베이스라인'과 **거의 같지 않은가?**
그렇다면 그때 모델은 아직 아무것도 못 배우고 **전부 '정상'이라고만 찍고 있던 것**이다.
정확도가 높아 보여도 실제로는 스팸을 하나도 못 잡고 있을 수 있다 — 그래서 재현율을 본다.

**(나) 위 '내 문자 시험'에서 놓친 게 있는가?**
계정 정지·링크 클릭 같은 **피싱 문구**를 정상으로 본다면, 이유를 생각해 보자.
이 데이터는 2005년 영국에서 모은 문자다 — 요즘 흔한 피싱 수법이 훈련 데이터에 거의 없었다면,
모델은 그걸 배울 기회가 없었다. **모델은 자기가 본 것만 안다.**

## 정리하며 생각할 것

1. **IMDB(0.68)보다 훨씬 잘 나왔다. 왜일까?**
   문자는 중앙값 12토큰, 영화 리뷰는 179토큰이었다.
   → 짧으면 마지막 은닉 상태까지 앞 내용이 살아서 도착한다. **오늘 §7의 뒷면이다.**
   RNN이 못 하는 게 아니라, **긴 문장**을 못 하는 것이다.
2. 그런데 정확도만 믿으면 안 된다는 걸 ④에서 봤다. 재현율은 어땠는가?
   스팸 필터라면 **놓친 스팸**과 **정상을 스팸함에 넣은 사고** 중 뭐가 더 치명적일까?
3. 오늘 만든 파이프라인을 또 어디에 붙일 수 있을까? (뉴스 분류, 문의 유형 분류 …)
"""
참고: wikidocs 10-02 스팸 메일 분류하기 https://wikidocs.net/22894
데이터 원본: UCI SMS Spam Collection (Almeida & Gómez Hidalgo, 2011)
"""